Импортируем Библиотеки

In [1]:
import pandas as pd
import numpy as np
from numba import njit

Функция для нахождения меньшей строки (по условию доминантности всех принадлежащих элементов)

In [2]:
@njit
def is_strictly_less(data_i, data_j):
    return np.all(data_i < data_j)

Фукция для сжатия датасета

In [3]:
def encode(data):
    min_vals = np.min(data, axis=0)
    max_vals = np.max(data, axis=0)

    range_vals = np.where(max_vals > min_vals, max_vals - min_vals, 1)
    normalized_data = (data - min_vals) / range_vals
    
    return normalized_data, min_vals, max_vals

Функция для разжатия датасета

In [4]:
def decode(encoded_data, min_vals, max_vals):
    range_vals = np.where(max_vals > min_vals, max_vals - min_vals, 1)
    restored_data = encoded_data * range_vals + min_vals
    
    return restored_data

Основная функция для нахождения доминирующих и доминируемых строк в датасете с учётом сжатия и расжатия (в нашем случае, устанавливаем индексацию на колонку "molecule_id")

In [ ]:
def find_strictly_less_relationships(file_path):
    try:
        df = pd.read_csv(file_path, index_col='molecule_id')
    except KeyError:
        raise ValueError("Column is missing in the dataset.")
    
    molecule_ids = df.index.values.flatten()
    
    numeric_columns = [col for col in df.columns if df[col].dtype in [np.int64, np.int32]]
    df = df[numeric_columns]
    
    df.fillna(0, inplace=True)
    
    data = df.to_numpy()
    n, m = data.shape
    
    encoded_data, min_vals, max_vals = encode(data)
    
    strictly_less_map_encoded = {molecule_id: [] for molecule_id in molecule_ids}
    
    for i in range(n):
        current_id = molecule_ids[i]
        
        for j in range(n):
            if i == j:
                continue
            
            if is_strictly_less(encoded_data[i], encoded_data[j]):
                strictly_less_map_encoded[current_id].append(molecule_ids[j])
    
    decoded_data = decode(encoded_data, min_vals, max_vals)
    
    strictly_less_map_decoded = {molecule_id: [] for molecule_id in molecule_ids}
    
    for i in range(n):
        current_id = molecule_ids[i]
        
        for j in range(n):
            if i == j:
                continue
            
            if is_strictly_less(decoded_data[i], decoded_data[j]):
                strictly_less_map_decoded[current_id].append(molecule_ids[j])
    
    result_encoded = {molecule_id: dominators for molecule_id, dominators in strictly_less_map_encoded.items() if dominators}
    result_decoded = {molecule_id: dominators for molecule_id, dominators in strictly_less_map_decoded.items() if dominators}
    
    return result_encoded, result_decoded

Выводим результаты

In [6]:
file_path = "./generated_dataset.csv"
result_encoded, result_decoded = find_strictly_less_relationships(file_path)

print("Результат на сжатых данных:")
print(result_encoded)

print("\nРезультат на распакованных данных:")
print(result_decoded)

Результат на сжатых данных:
{53: [6350, 9314], 89: [5126], 163: [2932], 185: [7862], 239: [6326], 298: [6326], 465: [675], 622: [2068], 686: [3928], 992: [6326], 1025: [4449], 1461: [5308], 1541: [9554], 1860: [3444], 2079: [6326], 2114: [5583, 6902], 2128: [6326, 7125], 2572: [1383, 4307, 9606], 2632: [5636], 2654: [3449, 6788], 3094: [675, 676], 3165: [675], 3269: [9641], 3457: [818, 2420, 6798], 3572: [676], 3574: [9362], 3617: [675], 3895: [8529], 3998: [6526], 4265: [9362], 4303: [676, 6459], 4484: [1904], 4647: [2130], 4735: [4789, 7500, 8312], 4967: [7862], 5199: [3658], 5220: [229, 6326], 5359: [3449], 5371: [510, 1785, 2277, 3658, 5126, 6326, 9183], 5526: [6114], 5717: [5853, 9873], 5914: [6902], 5939: [1084, 1710], 6158: [413, 4374, 7874], 6180: [2689, 8801], 6304: [1289], 6612: [9362], 6808: [675], 7233: [1904], 7352: [454, 1413, 1946], 7552: [7862], 7775: [675], 7913: [6959], 8111: [9362], 8139: [675, 6902], 8716: [6459], 8767: [6902], 8979: [675, 9606], 9013: [4245], 9031: